# Run Simulation — Evaluation & Visualization

Loads trained MAPPO weights and runs one deterministic evaluation episode.  
No training, no exploration — pure greedy policy.

### Files needed
```
uav_env.py
networks.py
simulation_log.csv
grid_config.json
actor0_final.pth
actor1_final.pth
```

## 1. Setup

In [ ]:
import os
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# ── Paths ─────────────────────────────────────────────────────────────────────
if os.path.exists('/kaggle/input'):
    BASE = '/kaggle/input/uav-crop-disease'
    sys.path.insert(0, BASE)
    WEIGHTS_DIR = '/kaggle/working'
else:
    BASE        = '..'
    WEIGHTS_DIR = '.'
    sys.path.insert(0, '.')

SIM_LOG_PATH  = os.path.join(BASE, 'simulation', 'simulation_log.csv') if not os.path.exists('/kaggle/input') else os.path.join(BASE, 'simulation_log.csv')
GRID_CFG_PATH = os.path.join(BASE, 'grid', 'grid_config.json')         if not os.path.exists('/kaggle/input') else os.path.join(BASE, 'grid_config.json')

from uav_env import UAVFieldEnv, GRID_ROWS, GRID_COLS, N_SECTORS, N_UAVS, E_MAX
from networks import ActorNetwork

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

## 2. Load Environment & Trained Weights

In [ ]:
env = UAVFieldEnv(SIM_LOG_PATH, GRID_CFG_PATH)

actors = [ActorNetwork().to(DEVICE) for _ in range(N_UAVS)]
for u in range(N_UAVS):
    path = os.path.join(WEIGHTS_DIR, f'actor{u}_final.pth')
    actors[u].load_state_dict(torch.load(path, map_location=DEVICE))
    actors[u].eval()
    print(f'Loaded: actor{u}_final.pth')

print('Ready for evaluation.')

## 3. Run One Evaluation Episode

In [ ]:
obs          = env.reset()
done         = False
total_reward = 0.0

# ── History for visualization ─────────────────────────────────────────────────
history = []

def snapshot(env, t, rewards):
    """Captures current environment state for visualization."""
    return {
        't'           : t,
        'uav_pos'     : list(env.uav_pos),
        'uav_status'  : env.uav_status.copy(),
        'true_status' : env.true_status.copy(),
        'risk_weights': env.w.copy(),
        'energy'      : list(env.energy),
        'rewards'     : rewards,
    }

# Save t=0 snapshot before any action
history.append(snapshot(env, 0, [0.0, 0.0]))

for t in range(env.T):
    actions = []
    for u in range(N_UAVS):
        obs_t  = torch.FloatTensor(obs[u]).to(DEVICE)
        action = actors[u].greedy_action(obs_t)   # deterministic, no sampling
        actions.append(action)

    obs, rewards, done, info = env.step(actions)
    total_reward += sum(rewards)
    history.append(snapshot(env, t + 1, rewards))

    if done:
        break

print(f'Episode complete — {len(history)} time steps')
print(f'Total reward     : {total_reward:.2f}')
print(f'Energy remaining : UAV0={env.energy[0]:.1f}  UAV1={env.energy[1]:.1f}')
n_discovered = int((env.uav_status != 2).sum())
n_infected   = int((env.uav_status == 1).sum())
print(f'Sectors visited  : {n_discovered} / {N_SECTORS}')
print(f'Infected found   : {n_infected}')

## 4. Grid Visualization — Frame by Frame

In [ ]:
ACTION_NAMES = ['Stay', 'North', 'South', 'West', 'East']
STATUS_LABEL = {0: 'H', 1: 'I', 2: '?'}   # healthy, infected, unknown
UAV_COLORS   = ['dodgerblue', 'darkorange']

os.makedirs('frames', exist_ok=True)

def plot_grid_frame(snap, save_path=None):
    t             = snap['t']
    uav_pos       = snap['uav_pos']
    uav_status    = snap['uav_status']
    true_status   = snap['true_status']
    risk_weights  = snap['risk_weights']
    energy        = snap['energy']

    fig, axes = plt.subplots(1, 2, figsize=(12, 5),
                             gridspec_kw={'width_ratios': [2, 1]})
    fig.suptitle(f'Day {t:02d} — UAV Field Monitoring', fontsize=13)

    # ── Left: Grid ────────────────────────────────────────────────────────────
    ax = axes[0]
    ax.set_xlim(-0.5, GRID_COLS - 0.5)
    ax.set_ylim(-0.5, GRID_ROWS - 0.5)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.set_xticks(range(GRID_COLS))
    ax.set_yticks(range(GRID_ROWS))
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    ax.set_title('Field Grid (H=Healthy  I=Infected  ?=Unknown)')

    norm = Normalize(vmin=0, vmax=1)
    cmap = plt.cm.RdYlGn_r   # red = high risk, green = low risk

    for sid in range(N_SECTORS):
        r, c     = sid // GRID_COLS, sid % GRID_COLS
        w        = risk_weights[sid]
        us       = uav_status[sid]
        ts       = true_status[sid]

        # Cell colour from risk weight
        color = cmap(norm(w))
        rect  = plt.Rectangle((c - 0.48, r - 0.48), 0.96, 0.96,
                               facecolor=color, edgecolor='black',
                               linewidth=1.5)
        ax.add_patch(rect)

        # UAV-known status label
        label = STATUS_LABEL[us]
        ax.text(c, r - 0.15, label, ha='center', va='center',
                fontsize=13, fontweight='bold',
                color='black')

        # Risk weight value
        ax.text(c, r + 0.25, f'w={w:.2f}', ha='center', va='center',
                fontsize=7, color='#333333')

        # Ground truth infection marker (star)
        if ts == 1:
            ax.text(c + 0.35, r - 0.35, '★', ha='center', va='center',
                    fontsize=9, color='darkred')

    # Draw UAV positions
    for u, (r, c) in enumerate(uav_pos):
        circle = plt.Circle((c, r), 0.22,
                             color=UAV_COLORS[u], zorder=5, alpha=0.85)
        ax.add_patch(circle)
        ax.text(c, r, f'U{u}', ha='center', va='center',
                fontsize=8, fontweight='bold', color='white', zorder=6)

    # Colorbar
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label='Risk Weight', fraction=0.03, pad=0.04)

    # ── Right: Stats panel ────────────────────────────────────────────────────
    ax2 = axes[1]
    ax2.axis('off')

    n_healthy  = int((uav_status == 0).sum())
    n_infected = int((uav_status == 1).sum())
    n_unknown  = int((uav_status == 2).sum())
    n_true_inf = int((true_status == 1).sum())

    stats = [
        f'Day          : {t}',
        '',
        f'UAV 0 pos    : {uav_pos[0]}',
        f'UAV 1 pos    : {uav_pos[1]}',
        '',
        f'Energy U0    : {energy[0]:.1f} / {E_MAX:.0f}',
        f'Energy U1    : {energy[1]:.1f} / {E_MAX:.0f}',
        '',
        '── UAV Knowledge ──',
        f'Healthy      : {n_healthy}',
        f'Infected     : {n_infected}',
        f'Unknown      : {n_unknown}',
        '',
        '── Ground Truth ──',
        f'True infected: {n_true_inf}',
        f'★ = infected sector',
    ]

    for i, line in enumerate(stats):
        ax2.text(0.05, 0.95 - i * 0.058, line,
                 transform=ax2.transAxes, fontsize=10,
                 fontfamily='monospace', va='top')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()

# Plot every 5 days to keep output manageable
for snap in history:
    if snap['t'] % 5 == 0:
        plot_grid_frame(snap, save_path=f"frames/grid_t{snap['t']:02d}.png")

print(f'Frames saved to frames/')

## 5. UAV Path Trajectories

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
ax.set_xlim(-0.5, GRID_COLS - 0.5)
ax.set_ylim(-0.5, GRID_ROWS - 0.5)
ax.set_aspect('equal')
ax.invert_yaxis()
ax.set_xticks(range(GRID_COLS))
ax.set_yticks(range(GRID_ROWS))
ax.grid(True, alpha=0.4)
ax.set_title('UAV Trajectories Over 30 Days', fontsize=13)
ax.set_xlabel('Column')
ax.set_ylabel('Row')

# Draw sector grid background
for sid in range(N_SECTORS):
    r, c = sid // GRID_COLS, sid % GRID_COLS
    rect = plt.Rectangle((c - 0.48, r - 0.48), 0.96, 0.96,
                          facecolor='#f0f0f0', edgecolor='gray',
                          linewidth=1)
    ax.add_patch(rect)
    ax.text(c, r, str(sid), ha='center', va='center',
            fontsize=9, color='gray', alpha=0.6)

# Mark initially infected sector
init_infected = [snap['true_status'] for snap in history if snap['t'] == 0][0]
for sid in range(N_SECTORS):
    if init_infected[sid] == 1:
        r, c = sid // GRID_COLS, sid % GRID_COLS
        ax.text(c, r - 0.3, '★', ha='center', fontsize=12, color='darkred')

# Draw UAV paths
for u in range(N_UAVS):
    path_r = [snap['uav_pos'][u][0] for snap in history]
    path_c = [snap['uav_pos'][u][1] for snap in history]

    # Line
    ax.plot(path_c, path_r, color=UAV_COLORS[u], linewidth=2,
            alpha=0.6, label=f'UAV {u}')

    # Arrows every 5 steps
    for i in range(0, len(path_r) - 1, 5):
        dr = path_r[i+1] - path_r[i]
        dc = path_c[i+1] - path_c[i]
        if dr != 0 or dc != 0:
            ax.annotate('', xy=(path_c[i+1], path_r[i+1]),
                        xytext=(path_c[i], path_r[i]),
                        arrowprops=dict(arrowstyle='->', color=UAV_COLORS[u],
                                        lw=1.5))

    # Start and end markers
    ax.plot(path_c[0],  path_r[0],  'o', color=UAV_COLORS[u],
            markersize=10, label=f'UAV {u} start')
    ax.plot(path_c[-1], path_r[-1], 's', color=UAV_COLORS[u],
            markersize=10, label=f'UAV {u} end')

ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('uav_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: uav_trajectories.png')

## 6. Risk Field & Coverage Over Time

In [ ]:
timesteps    = [s['t'] for s in history]
n_discovered = [(s['uav_status'] != 2).sum() for s in history]
n_infected_found = [(s['uav_status'] == 1).sum() for s in history]
energy_u0    = [s['energy'][0] for s in history]
energy_u1    = [s['energy'][1] for s in history]
mean_risk    = [s['risk_weights'].mean() for s in history]
true_infected= [int((s['true_status'] == 1).sum()) for s in history]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('Evaluation Episode — Summary Statistics', fontsize=13)

# Coverage
axes[0,0].plot(timesteps, n_discovered, color='steelblue', linewidth=2)
axes[0,0].axhline(N_SECTORS, color='gray', linestyle='--', label='Total sectors')
axes[0,0].set_title('Sectors Visited Over Time')
axes[0,0].set_xlabel('Day')
axes[0,0].set_ylabel('Sectors')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Infected found vs actual
axes[0,1].plot(timesteps, true_infected, color='tomato',
               linewidth=2, linestyle='--', label='True infected (ground truth)')
axes[0,1].plot(timesteps, n_infected_found, color='darkred',
               linewidth=2, label='Infected found by UAVs')
axes[0,1].set_title('Infected Sectors: Found vs Actual')
axes[0,1].set_xlabel('Day')
axes[0,1].set_ylabel('Sectors')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Energy
axes[1,0].plot(timesteps, energy_u0, color=UAV_COLORS[0],
               linewidth=2, label='UAV 0')
axes[1,0].plot(timesteps, energy_u1, color=UAV_COLORS[1],
               linewidth=2, label='UAV 1')
axes[1,0].set_title('Energy Remaining per UAV')
axes[1,0].set_xlabel('Day')
axes[1,0].set_ylabel('Energy units')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Mean risk weight
axes[1,1].plot(timesteps, mean_risk, color='darkorchid', linewidth=2)
axes[1,1].set_title('Mean Risk Weight Across Field')
axes[1,1].set_xlabel('Day')
axes[1,1].set_ylabel('Mean w[k]')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: evaluation_summary.png')

## 7. Final Summary

In [ ]:
final = history[-1]
n_vis = int((final['uav_status'] != 2).sum())
n_inf = int((final['uav_status'] == 1).sum())
n_tru = int((final['true_status'] == 1).sum())

print('=' * 45)
print('        EVALUATION SUMMARY')
print('=' * 45)
print(f'  Total days simulated  : {final["t"]}')
print(f'  Total reward          : {total_reward:.2f}')
print(f'  Sectors visited       : {n_vis} / {N_SECTORS}')
print(f'  Infected sectors found: {n_inf}')
print(f'  True infected (final) : {n_tru}')
print(f'  Detection rate        : {n_inf/max(n_tru,1)*100:.1f}%')
print(f'  Energy left UAV 0     : {final["energy"][0]:.1f} / {E_MAX:.0f}')
print(f'  Energy left UAV 1     : {final["energy"][1]:.1f} / {E_MAX:.0f}')
print('=' * 45)